In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
from tablevault import tablevault
import os

vault = tablevault.Vault(user_id="jinjin",
                            process_name="distilbert_max_pool_cosine_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

print(model_name)
print("hidden_size:", model.config.hidden_size)


---[ TableVault Record ]---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


distilbert-base-uncased
hidden_size: 768
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [6]:
def masked_max_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand_as(last_hidden_state).bool()
    masked_hidden = last_hidden_state.masked_fill(~mask, -1e9)
    pooled = masked_hidden.max(dim=1).values
    pooled = torch.nn.functional.normalize(pooled, p=2, dim=-1)
    return pooled


def encode_texts(texts, batch_size=128, max_length=128):
    all_embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size)):
            batch_texts = texts[i:i + batch_size]
            enc = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            outputs = model(**enc)
            pooled = masked_max_pool(outputs.last_hidden_state, enc["attention_mask"])
            all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0)


test_hidden = torch.randn(2, 4, 8)
test_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]])
test_pooled = masked_max_pool(test_hidden, test_mask)
print("test pooled shape:", tuple(test_pooled.shape))


---[ TableVault Record ]---
test pooled shape: (2, 8)
---[ TableVault Record ]---



In [7]:
batch_size = 128
threshold = 0.85

emb1 = encode_texts(sent1, batch_size=batch_size)
emb2 = encode_texts(sent2, batch_size=batch_size)

cosine_scores = (emb1 * emb2).sum(dim=-1).numpy()
y_pred = (cosine_scores >= threshold).astype(int)

print("emb1:", tuple(emb1.shape))
print("emb2:", tuple(emb2.shape))
print("done")


---[ TableVault Record ]---


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

emb1: (408, 768)
emb2: (408, 768)
done
---[ TableVault Record ]---



In [8]:

vault.create_record_list("cosine_mrpc_max_pool_similarity_prediction", column_names=["prediction", "cosine_scores"])

for i in range(len(y_pred)):
    vault.append_record("cosine_mrpc_max_pool_similarity_prediction", 
                        {
                            "prediction": int(y_pred[i]),
                            "cosine_scores": float(cosine_scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "This dataset stores per-example paraphrase predictions for the GLUE MRPC validation set produced by a DistilBERT-based cosine similarity pipeline. For each sentence pair from glue_mrpc_validation, the notebook encodes sentence1 and sentence2 with distilbert-base-uncased, applies masked max pooling to the token embeddings, computes the cosine similarity between the two normalized sentence embeddings, and converts that score to a binary prediction using a threshold of 0.85.\n\nStructure:\n- prediction: integer binary label predicted by the similarity rule (1 = paraphrase, 0 = not paraphrase)\n- cosine_scores: float cosine similarity between the max-pooled embeddings of the two sentences\n\nEach record corresponds to one input sentence pair and is linked back to the source validation examples through input_items. In this workflow, the dataset serves as the main model-output table for downstream evaluation, inspection of correct and incorrect predictions, and creation of the final summary metrics dataset."
embedding = get_embeddings(description)
vault.create_description("cosine_mrpc_max_pool_similarity_prediction", description, embedding)

properties = {"task": "paraphrase detection", "artifact_type": "prediction dataset", "prediction_type": "binary classification", "score_type": "cosine similarity", "pooling": "masked max pooling", "embedding_model": "distilbert-base-uncased", "similarity_threshold": "0.85", "input_type": "sentence pair", "source": "glue/mrpc", "dataset": "MRPC", "split": "validation", "size": "408", "label_space": "0=not_paraphrase, 1=paraphrase", "text_fields": "sentence1,sentence2"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("cosine_mrpc_max_pool_similarity_prediction", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [9]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])
print({"accuracy": acc, "f1": f1, "threshold": threshold})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


---[ TableVault Record ]---
{'accuracy': 0.6838235294117647, 'f1': 0.8122270742358079, 'threshold': 0.85}
                precision    recall  f1-score   support

not_paraphrase       0.00      0.00      0.00       129
    paraphrase       0.68      1.00      0.81       279

      accuracy                           0.68       408
     macro avg       0.34      0.50      0.41       408
  weighted avg       0.47      0.68      0.56       408

---[ TableVault Record ]---



/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jinjinzhao/Documents/work_projects/tablevault_experiments/tablevault_experiments/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Prec

In [10]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("cosine:", float(cosine_scores[i]))
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))


---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
cosine: 0.9697094559669495
true: 1 pred: 1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
cosine: 0.9531527161598206
true: 0 pred: 1
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
cosine: 0.9761152267456055
true: 0 pred: 1
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO anno

In [11]:
vault.create_record_list("distilbert_max_pool_cosine_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("distilbert_max_pool_cosine_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "cosine_mrpc_max_pool_similarity_prediction": [0, len(ds)]
                    })

summary

description = "This dataset stores the experiment-level evaluation summary for the DistilBERT max-pooling cosine-similarity approach on the GLUE MRPC validation set. It contains a single summary record with three fields: accuracy (float), f1 (float), and classification_report (string). The values are computed by comparing binary paraphrase predictions\u2014generated by thresholding cosine similarity between max-pooled DistilBERT embeddings of sentence pairs\u2014against the ground-truth MRPC labels. In this workflow, the dataset serves as the final aggregated results table for the notebook, capturing overall model performance in a compact form for later inspection, comparison, and lineage tracking back to the validation data and prediction outputs."
embedding = get_embeddings(description)
vault.create_description("distilbert_max_pool_cosine_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "408", "model": "distilbert-base-uncased", "embedding_pooling": "masked max pooling", "similarity_metric": "cosine similarity", "decision_rule": "threshold-based binary classification", "threshold": "0.85", "inputs": "sentence pair", "outputs": "accuracy, f1, classification report", "process": "distilbert_max_pool_cosine_mrpc"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_max_pool_cosine_mrpc_summary", cat, embedding, prop)




---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook evaluates a simple paraphrase detection pipeline on the GLUE MRPC validation set using DistilBERT sentence embeddings. It loads sentence pairs and labels from TableVault, encodes each sentence with distilbert-base-uncased, applies masked max pooling over token embeddings, L2-normalizes the pooled vectors, and computes cosine similarity between the two sentence embeddings. A fixed cosine threshold is used to convert similarity scores into binary paraphrase predictions, and the notebook reports accuracy, F1, a full classification report, and example errors. It also records per-example predictions, summary metrics, and workflow metadata back into TableVault, with OpenAI-generated description embeddings for searchable experiment documentation." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("distilbert_max_pool_cosine_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary sentence-pair classification", "dataset": "glue/mrpc", "data_split": "validation", "model": "distilbert-base-uncased", "embedding_strategy": "DistilBERT token embeddings with masked max pooling", "similarity_metric": "cosine similarity", "decision_rule": "threshold-based classification", "threshold": "0.85", "framework": "PyTorch, Hugging Face Transformers", "evaluation": "accuracy, f1-score, classification_report", "workflow": "encode sentence pairs, compute cosine similarity, threshold predictions, error analysis", "tracking": "TableVault", "notebook_type": "baseline semantic similarity evaluation"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert_max_pool_cosine_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

